In [ ]:
# NLTK download SSL: Certificate verify failed
'''
nltk.download('punkt')    
 [nltk_data] Error loading Punkt: <urlopen error [SSL:
 [nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed
 [nltk_data]     (_ssl.c:)>
False
'''

import ssl  # Secure Sockets Layer (SSL) for encrypted communication
import nltk  # Natural Language Toolkit (NLTK) for text processing

# This block **disables SSL certificate verification** for this Python session.
# Needed when Python cannot verify SSL certificates (common in Homebrew installs on macOS).
try:
    _create_unverified_https_context = ssl._create_unverified_context  # Create an unverified SSL context

except AttributeError:
    # Some older Python versions may not support `_create_unverified_context`, so we handle the exception.
    pass  

else:
    # If the SSL context is successfully created, override Python’s default SSL verification.
    ssl._create_default_https_context = _create_unverified_https_context  

# Now that SSL verification is disabled, attempt to download the `punkt` dataset.
# `punkt` is required for **sentence tokenization** in NLTK.
nltk.download('punkt')

[nltk_data] Downloading package punkt to /Users/BAEK/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [1]:
# Previously, langchain included all integrations (OpenAI, HuggingFace, Chroma, Pinecone, etc.), making it very large.
# Now, the core langchain package is lighter, and specific integrations are separate optional dependencies.

# DirectoryLoader is a document loader from LangChain.
# It loads multiple text documents from a folder at once.
# from langchain.document_loaders import DirectoryLoader
from langchain_community.document_loaders import DirectoryLoader

# This is a text-splitting tool that breaks large documents into smaller chunks.
# It ensures that chunks remain meaningful by breaking at the largest possible separator (like paragraphs before sentences).
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Document is a data structure used in LangChain. 
# This helps track where each text chunk comes from, useful for debugging and retrieval.
from langchain.schema import Document

# from langchain.embeddings import OpenAIEmbeddings
# If you use OpenAI, you only need langchain_openai.
from langchain_openai import OpenAIEmbeddings

# from langchain.vectorstores import Chroma
# If you use Chroma, you only need langchain_community.
from langchain_community.vectorstores import Chroma

# This library connects to OpenAI’s API, allowing interaction with GPT-4, embeddings, and other models.
import openai

# dotenv allows us to store and load environment variables (like API keys) from a .env file.
from dotenv import load_dotenv

# The built-in os module allows interaction with the operating system, like handling file paths.
# Needed for loading files from directories, managing file paths, and accessing environment variables.
import os

# The shutil module provides high-level file operations, like copying, moving, or deleting directories.
# Needed when organizing large text datasets, cleaning up files, or moving processed documents.
import shutil

In [4]:
# Load environment variables. Assumes that project contains .env file with API keys
load_dotenv()
#---- Set OpenAI API key 
# Change environment variable name from "OPENAI_API_KEY" to the name given in 
# your .env file.
openai.api_key = os.environ['OPENAI_API_KEY']

# A setting for the database directory where ChromaDB will store and persist its vector embeddings. 
# You do not need to manually set up ChromaDB in advance—it will be created automatically when you run your code.
CHROMA_PATH = "chroma"
DATA_PATH = "data/books"

In [ ]:
# Defines a function named load_documents() that loads all Markdown files from a specified directory.
def load_documents():
    
    # DATA_PATH → This is the directory path where Markdown (.md) files are stored.
	# glob="*.md" → This ensures that only .md files are loaded (excluding other file types like .txt or .pdf).
	# The DirectoryLoader class scans the DATA_PATH directory and collects all files that match the glob pattern.
    loader = DirectoryLoader(DATA_PATH, glob = "*.md")
    
    documents = loader.load()
    
    return documents


In [24]:
loader = DirectoryLoader(DATA_PATH, glob="*.md")

documents = loader.load()

print("LENGTH: ", len(documents))

LENGTH:  1


In [ ]:
# The function takes a list of Document objects as input.
# The Document type is a LangChain schema that stores text content and metadata.
# The input documents could be large texts (books, reports, articles, etc.).
def split_text(documents: list[Document]):
    
    '''
    ✅ Increase chunk_size if:
        •	You need full sentences or paragraphs in a single chunk.
        •	Your document is highly structured (e.g., books, research papers).
        •	The chunks are not retrieving enough relevant content.

    ✅ Decrease chunk_size if:
        •	You need more granular retrieval (e.g., FAQs, short articles).
        •	The LLM has token limitations (some models struggle with long inputs).
        •	The dataset contains many small, independent pieces of text.

    ✅ Increase chunk_overlap if:
        •	Chunks lack continuity when retrieving text.
        •	The model struggles to answer questions due to missing context.
        •	You have highly technical or structured data (e.g., legal, medical).

    ✅ Decrease chunk_overlap if:
        •	Chunks contain too much duplicate text, leading to wasted token space.
        •	The dataset is very large, and reducing redundancy improves storage efficiency
    '''
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 300,      # The number of characters in each text chunk.
        chunk_overlap = 100,   # EThe overlapping region (in characters) between chunks.
        length_function = len, # Measures text length using the built-in `len()` function
        add_start_index = True # Adds an index to track where each chunk starts
    )
    
    # if you pass multiple documents to split_text(documents), the function processes them all and returns a single list of chunks, effectively combining all chunks from different documents.
    chunks = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(chunks)} chunks.")

    document = chunks[10]
    
    print(document.page_content)
    print(document.metadata)

    return chunks

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=100,
    length_function=len,
    add_start_index=True,
)
    
chunks = text_splitter.split_documents(documents)
print("CHUNKS:", chunks)

print(f"Split {len(documents)} documents into {len(chunks)} chunks.")

document_index_i = chunks[1]

print("PAGE_CONTENTS: ", document_index_i.page_content)
print("METADATA: ", document_index_i.metadata)

CHUNKS: [Document(metadata={'source': 'data/books/alice_in_wonderland.md', 'start_index': 0}, page_content="The Project Gutenberg eBook of Alice's Adventures in Wonderland"), Document(metadata={'source': 'data/books/alice_in_wonderland.md', 'start_index': 65}, page_content='This ebook is for the use of anyone anywhere in the United States and\nmost other parts of the world at no cost and with almost no restrictions\nwhatsoever. You may copy it, give it away or re-use it under the terms\nof the Project Gutenberg License included with this ebook or online'), Document(metadata={'source': 'data/books/alice_in_wonderland.md', 'start_index': 279}, page_content='of the Project Gutenberg License included with this ebook or online\nat www.gutenberg.org. If you are not located in the United States,\nyou will have to check the laws of the country where you are located\nbefore using this eBook.'), Document(metadata={'source': 'data/books/alice_in_wonderland.md', 'start_index': 509}, page_content="

In [ ]:
# Function accepts a list of document chunks as input.
def save_to_chroma(chunks: list[Document]):  
    
    # Clear out the existing Chroma database if it exists.
    # This step ensures you're working with a fresh database by removing old data.
    if os.path.exists(CHROMA_PATH):  
        # If it exists, remove the entire folder (database files).
        shutil.rmtree(CHROMA_PATH)      

    # Create a new Chroma database from the given document chunks.
    # Each chunk will be embedded and stored in the database.
    db = Chroma.from_documents(
        chunks,                         # The document chunks (processed text data).
        OpenAIEmbeddings(),             # Use OpenAI's embedding model to convert text into vector embeddings.
        persist_directory = CHROMA_PATH # Directory where the Chroma database will be stored.
    )
    
    # Chroma 0.4.x and above no longer require you to manually call persist().
    # Persist (save) the database so that it remains stored on disk.
    # Without this, the data would only exist in memory, and after the program ends, it would be lost. db.persist() ensures that all embedded data is saved to disk.
    #db.persist()
    
    print(f"Saved {len(chunks)} chunks to {CHROMA_PATH}.")

In [ ]:
if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)

db = Chroma.from_documents(
    chunks, OpenAIEmbeddings(), persist_directory=CHROMA_PATH
)

# db.persist()

print(f"Saved {len(chunks)} chunks to {CHROMA_PATH}.")

Saved 801 chunks to chroma.


/var/folders/dh/551p960n6mv9t9byzj8pp0640000gp/T/ipykernel_63631/409866526.py:8: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


In [ ]:
# Acts as the entry point of the program, calling other functions.
def main():
    generate_data_store()

# Contains the logic for processing data: loading documents, splitting them into chunks, and saving them to the database.
def generate_data_store():
    
    # Loads the documents from the specified directory.
    documents = load_documents()
    
    # Splits the loaded documents into chunks for further processing.
    chunks = split_text(documents)
    
    # Saves the chunks as embeddings in the Chroma database.
    save_to_chroma(chunks)
    
# This modularity allows the code to be easier to understand, maintain, and test. 
# You can easily modify or debug individual parts without affecting the entire program.

In [ ]:
# Control how the script is executed when the file is run.

# How Does it Work?
# 1. When you run a Python script directly (e.g., python script.py), the value of __name__ is set to "__main__".
# 2. When you import the script into another module, the value of __name__ is set to the name of the script/module (e.g., script if the file is script.py).

# Why Use if __name__ == "__main__":?
# 1. To ensure code is only executed when the script is run directly, and not when imported as a module.
# 2. Without this, if you import this file in another script, all functions and the logic inside main() would run automatically, which is not usually desired.

# When you import a Python file (module) into another script, you typically only want to access specific functions or classes, not execute the whole program.
# Why if __name__ == "__main__": Helps
# • Prevents unwanted execution: It ensures that only the relevant functions are accessible when the module is imported, and it doesn’t run the script-specific code.
# • Selective function usage: You can now import the script and call specific functions without worrying about other code executing.
if __name__ == "__main__":
    main()

# Without the if __name__ == "__main__": block, load_documents() would be executed along with the rest of the script when you import it, which is not desired.